# BSH Conjugation Model

In [ ]:
# stdlib
import os, re, math, unicodedata
from collections import Counter
from difflib import get_close_matches

# data
import numpy as np
import pandas as pd
import h5py

# chemistry
from rdkit import Chem
from rdkit.Chem import AllChem, Draw

# deep learning
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit

# paths
from pathlib import Path

DATA_DIR   = Path("../data")
OUTPUT_DIR = Path("../outputs")

## Loading Embeddings

In [ ]:
H5_PATH = DATA_DIR / "Seqs_list_total.h5"

heat_long = pd.read_csv(OUTPUT_DIR / "ipsita_heatmap_long.csv")

# ---- helpers to enumerate H5 datasets ----
def collect_h5_datasets(h5_path):
    paths = []
    with h5py.File(h5_path, "r") as f:
        def visit(name, obj):
            if isinstance(obj, h5py.Dataset):
                paths.append(name)
        f.visititems(visit)
    return paths

def norm_key(s: str) -> str:
    return re.sub(r"[^A-Za-z0-9]", "", str(s)).upper()

def last_token(code: str) -> str:
    return str(code).split("_")[-1]

# ---- scan H5 once and index by normalized base name ----
all_paths = collect_h5_datasets(H5_PATH)
if not all_paths:
    raise RuntimeError(f"No datasets found in {H5_PATH}.")

bases = [p.rsplit("/", 1)[-1] for p in all_paths]
norm_bases = [norm_key(b) for b in bases]
normbase_to_path = {}
for base, normb, full in zip(bases, norm_bases, all_paths):
    normbase_to_path.setdefault(normb, full)

all_norm_bases = list(normbase_to_path.keys())

def pick_contains_match(tok_norm: str):
    hits = [b for b in all_norm_bases if tok_norm in b or b in tok_norm]
    if not hits: return None
    hits = sorted(hits, key=len)
    return normbase_to_path[hits[0]]

def pick_fuzzy_match(tok_norm: str, cutoff=0.92):
    cand = get_close_matches(tok_norm, all_norm_bases, n=1, cutoff=cutoff)
    return (normbase_to_path[cand[0]] if cand else None)

def match_dataset_for_enzyme(enzyme_id: str) -> str | None:
    full_norm = norm_key(enzyme_id)
    tok_norm  = norm_key(last_token(enzyme_id))

    if full_norm in normbase_to_path:
        return normbase_to_path[full_norm]
    if tok_norm in normbase_to_path:
        return normbase_to_path[tok_norm]
    p = pick_contains_match(tok_norm)
    if p: return p
    return pick_fuzzy_match(tok_norm, cutoff=0.92)

def fetch_and_pool(h5_path, full_path):
    with h5py.File(h5_path, "r") as f:
        arr = np.array(f[full_path])
    if arr.ndim == 2 and arr.shape[0] > 1:
        arr = arr.mean(axis=0)
    elif arr.ndim == 2 and arr.shape[0] == 1:
        arr = arr[0]
    elif arr.ndim > 2:
        arr = arr.reshape(arr.shape[-1])
    return arr.astype(np.float32)

# ---- do the matching for enzymes present in heat_long ----
enz_ids = sorted(heat_long["Enzyme"].astype(str).unique().tolist())
rows, unmatched = [], []
for eid in enz_ids:
    path = match_dataset_for_enzyme(eid)
    if path is None:
        unmatched.append(eid); continue
    try:
        vec = fetch_and_pool(H5_PATH, path)
        rows.append((eid, path, vec, int(vec.shape[-1])))
    except Exception as e:
        print(f"[Warn] failed to read {eid} at {path}: {e}")

if not rows:
    raise RuntimeError("No embeddings loaded\u2014check H5 structure and enzyme naming.")

df_enz = pd.DataFrame(rows, columns=["Enzyme", "h5_path", "Embedding", "dim"])
E = np.stack(df_enz["Embedding"].values, axis=0).astype(np.float32)
d_prot = E.shape[1]
enzyme2idx = {eid: i for i, eid in enumerate(df_enz["Enzyme"].tolist())}

print(f"[OK] Embedding matrix: {E.shape} (d_prot={d_prot})")
print(f"[Match] {len(df_enz)} matched / {len(enz_ids)} enzymes in heat_long.")
if unmatched[:10]:
    print("[Unmatched] examples:", unmatched[:10])

np.save(OUTPUT_DIR / "enzyme_embeddings.npy", {row.Enzyme: row.Embedding for _, row in df_enz.iterrows()})
emb_cols = [f"enz_{i}" for i in range(d_prot)]
pd.concat(
    [df_enz[["Enzyme"]].reset_index(drop=True),
     pd.DataFrame(E, columns=emb_cols)],
    axis=1
).to_csv(OUTPUT_DIR / "enzyme_embeddings.csv", index=False)
print("Saved: enzyme_embeddings.npy and enzyme_embeddings.csv")

In [ ]:
df_enz.head()

## Creating Path to Keep Track of All Possible Pairs

1. `substrates_catalog_pairs.csv` \u2014 the dictionary of all unique BA\u2013Amine combinations (one row per pair with names/SMILES and a stable feat_idx).
2. `group_membership_with_featidx.csv` \u2014 the index that maps each ProductName to its dictionary entry via sub_key and feat_idx.
3. `substrate_features_pairs.npz` \u2014 fingerprint for each dictionary entry.

In [ ]:
ENUM_XLSX = OUTPUT_DIR / "swap_enumeration_FINAL.xlsx"
ENUM_CSV  = OUTPUT_DIR / "swap_enumeration_FINAL.csv"

if os.path.exists(ENUM_XLSX):
    enum_df = pd.read_excel(ENUM_XLSX)
elif os.path.exists(ENUM_CSV):
    enum_df = pd.read_csv(ENUM_CSV)
else:
    raise FileNotFoundError("swap_enumeration_FINAL.(xlsx/csv) not found.")

for c in ["ProductName","Parent_BA_Name","Parent_BA_SMILES_Original",
          "Acid_SMILES_Used","Amine_Name","Amine_SMILES","Note"]:
    if c not in enum_df.columns:
        enum_df[c] = np.nan

# ---------- helpers ----------
def split_products(s: str):
    if pd.isna(s): return []
    parts = [p.strip() for p in str(s).split(";") if str(p).strip()]
    return parts if parts else []

def pick_ba_smiles(row):
    for c in ["Parent_BA_SMILES_Original","Acid_SMILES_Used"]:
        smi = str(row.get(c, "") or "").strip()
        if smi: return smi
    return ""

def morgan_fp(smi, nBits=1024, radius=2):
    if not smi: return np.zeros(nBits, dtype=np.float32)
    m = Chem.MolFromSmiles(str(smi))
    if m is None: return np.zeros(nBits, dtype=np.float32)
    bv = AllChem.GetMorganFingerprintAsBitVect(m, radius=radius, nBits=nBits)
    arr = np.zeros((nBits,), dtype=np.int8)
    Chem.DataStructs.ConvertToNumpyArray(bv, arr)
    return arr.astype(np.float32)

def name_hash_fp(name: str, nBits=256):
    s = (str(name) or "").lower().strip()
    grams = set(); t = f"^{s}$"
    for i in range(max(0, len(t)-2)):
        grams.add(t[i:i+3])
    bits = np.zeros(nBits, dtype=np.float32)
    for g in grams:
        idx = (hash(g) % nBits + nBits) % nBits
        bits[idx] = 1.0
    return bits

# ---------- 1) membership: (ProductName, sub_key) ----------
rows = []
for _, r in enum_df.iterrows():
    pnames = split_products(r["ProductName"])
    pname  = pnames if pnames else [str(r["ProductName"]).strip()]
    ba_nm  = str(r["Parent_BA_Name"] or "").strip()
    am_nm  = str(r["Amine_Name"] or "").strip()
    if not ba_nm and not am_nm:
        continue
    sub_key = f"{ba_nm} || {am_nm}"
    for pn in pname:
        if pn:
            rows.append((pn, sub_key))
membership = pd.DataFrame(rows, columns=["ProductName","sub_key"]).drop_duplicates().reset_index(drop=True)
print(f"[membership] products={membership['ProductName'].nunique()}  rows={len(membership)}")

# ---------- 2) substrates_catalog_pairs: unique BA||Amine with SMILES ----------
rows = []
seen = set()
for _, r in enum_df.iterrows():
    ba_nm = str(r["Parent_BA_Name"] or "").strip()
    am_nm = str(r["Amine_Name"] or "").strip()
    if not ba_nm and not am_nm:
        continue
    sk = f"{ba_nm} || {am_nm}"
    if sk in seen:
        continue
    seen.add(sk)
    rows.append({
        "sub_key": sk,
        "parent_ba_name": ba_nm,
        "amine_name": am_nm,
        "ba_smiles": pick_ba_smiles(r),
        "amine_smiles": (str(r["Amine_SMILES"]) or "").strip(),
    })
substrates = pd.DataFrame(rows).reset_index(drop=True)

substrates["feat_idx"] = np.arange(len(substrates), dtype=int)
subkey2idx = dict(zip(substrates["sub_key"], substrates["feat_idx"]))
membership["feat_idx"] = membership["sub_key"].map(subkey2idx)

# ---------- 3) compute features ----------
A_bits, B_bits = [], []
nA_smi = nA_hash = nB_smi = nB_hash = 0
for _, r in substrates.iterrows():
    a_smi = (r["amine_smiles"] or "").strip()
    b_smi = (r["ba_smiles"] or "").strip()

    if a_smi:
        A = morgan_fp(a_smi, nBits=1024, radius=2); nA_smi += 1
    else:
        A = name_hash_fp(r["amine_name"], nBits=1024);   nA_hash += 1

    if b_smi:
        B = morgan_fp(b_smi, nBits=512, radius=2);       nB_smi += 1
    else:
        B = name_hash_fp(r["parent_ba_name"], nBits=512); nB_hash += 1

    A_bits.append(A); B_bits.append(B)

A_bits = np.stack(A_bits, axis=0).astype(np.float32)
B_bits = np.stack(B_bits, axis=0).astype(np.float32)

print(f"[features] amine: SMILES {nA_smi} | hash {nA_hash}  -> {A_bits.shape}")
print(f"[features] BA   : SMILES {nB_smi} | hash {nB_hash}  -> {B_bits.shape}")

# ---------- 4) save artifacts ----------
membership.to_csv(OUTPUT_DIR / "group_membership_with_featidx.csv", index=False)
substrates.to_csv(OUTPUT_DIR / "substrates_catalog_pairs.csv", index=False)
np.savez_compressed(
    OUTPUT_DIR / "substrate_features_pairs.npz",
    A=A_bits, B=B_bits,
    sub_keys=substrates["sub_key"].values,
    parent_ba_names=substrates["parent_ba_name"].values,
    amine_names=substrates["amine_name"].values,
    feat_idx=substrates["feat_idx"].values,
)
print("Saved: group_membership_with_featidx.csv, substrates_catalog_pairs.csv, substrate_features_pairs.npz")

Substrates: Accounts for the bile acid + amine SMILES that account for the product.

In [ ]:
substrates[substrates.sub_key.str.contains('alanine')]

Index table that goes back to the substrate table.

In [ ]:
temp = 'Tri_'
membership[membership.ProductName.str.contains(temp)].head(30)

## Multiple Instance Learning Model

Which bile acid would be the best for the BA+Amine pair?

**Inputs:** E = enzyme embedding, A = amine fingerprint, B = bile-acid fingerprints

**Scorer (MLP):** Determines which bile acid is most compatible with the specific amine

**Mixer (attention):** Computes weights over the BA set

In [ ]:

CFG = dict(
    seed=1337,
    device="cuda" if torch.cuda.is_available() else "cpu",
    label_log1p=True,           # ✅ log1p targets for LC-MS scales
    batch_size=256,
    epochs=40,
    lr=3e-4,
    weight_decay=1e-4,
    grad_clip=1.0,
    lambda_entropy=1e-3,        # encourages sparse attention (note: sign discussed earlier)
    val_frac=0.15,
    test_frac=0.15,
    patience=8,
    min_delta=1e-4,
    ckpt_path="best_mil.pt",
    topk_show=5,
)

# ================== utils ==================
def set_seeds(seed:int):
    import random
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def rmse(a,b): a,b=np.asarray(a),np.asarray(b); return float(np.sqrt(np.mean((a-b)**2)))
def mae(a,b):  a,b=np.asarray(a),np.asarray(b); return float(np.mean(np.abs(a-b)))
def pearsonr(a,b):
    a,b=np.asarray(a),np.asarray(b)
    if a.std()==0 or b.std()==0: return 0.0
    return float(np.corrcoef(a,b)[0,1])

# ================== re-use your in-memory objects ==================
# Use existing enzyme embeddings & index map
if 'E_mat' not in globals():
    if 'E' in globals():
        E_mat = E  # from your earlier H5 step
    else:
        raise NameError("Neither E_mat nor E found. Please define enzyme embeddings first.")

assert 'enzyme2idx' in globals(), "enzyme2idx not found. Provide dict enzyme_id -> row index in E_mat."

# Use existing heat_long, else try to load (xlsx or csv)
if 'heat_long' not in globals():
    src = str(OUTPUT_DIR / "ipsita_heatmap_long.csv")
    if not os.path.exists(src):
        raise FileNotFoundError("heat_long not in memory and ipsita_heatmap_long.csv not found.")
    heat_long = pd.read_csv(src)

# ================== labels: average triplicates ==================
df_hl = heat_long.copy()
# standardize columns the pipeline expects
ren = {}
if "Enzyme" in df_hl.columns:      ren["Enzyme"] = "enzyme_id"
if "ProductName" in df_hl.columns: ren["ProductName"] = "group_id"
if "Intensity" in df_hl.columns:   ren["Intensity"] = "y"
df_hl = df_hl.rename(columns=ren)

need = {"enzyme_id","group_id","y"}
missing_cols = need - set(df_hl.columns)
if missing_cols:
    raise AssertionError(f"`heat_long` must have {need}. Missing: {missing_cols}. Got: {df_hl.columns.tolist()}")

labels_long = (
    df_hl[["enzyme_id","group_id","y"]]
      .assign(y=lambda d: pd.to_numeric(d["y"], errors="coerce"))
      .dropna(subset=["y"])
      .groupby(["enzyme_id","group_id"], as_index=False)["y"].mean()
)
print(f"[labels] rows={len(labels_long)} enzymes={labels_long['enzyme_id'].nunique()} groups={labels_long['group_id'].nunique()}")

# (optional) strip replicate suffixes if embeddings don’t include them
def strip_rep_suffix(s: str) -> str:
    return str(s).rsplit("_rep", 1)[0]
labels_long["enzyme_id"] = labels_long["enzyme_id"].astype(str).map(strip_rep_suffix)

# ================== load prebuilt membership & features ==================
MEM_CSV = str(OUTPUT_DIR / "group_membership_with_featidx.csv")
SUB_CSV = str(OUTPUT_DIR / "substrates_catalog_pairs.csv")
PAIR_NPZ = str(OUTPUT_DIR / "substrate_features_pairs.npz")

assert os.path.exists(MEM_CSV) and os.path.exists(SUB_CSV) and os.path.exists(PAIR_NPZ), \
    "Expected saved artifacts (membership/features) not found. Run your earlier enumeration block once."

mem = pd.read_csv(MEM_CSV)   # columns: ProductName, sub_key, feat_idx
sub = pd.read_csv(SUB_CSV)   # columns: feat_idx, parent_ba_name, amine_name, ba_smiles, amine_smiles
pairs_npz = np.load(PAIR_NPZ, allow_pickle=True)
A = pairs_npz["A"].astype(np.float32)                       # [N_subkeys, 1024] amine features
B = pairs_npz["B"].astype(np.float32)                       # [N_subkeys,  512] bile-acid features
feat_idx = pairs_npz["feat_idx"].astype(int)                # [N_subkeys]
idx2row = {int(fi): i for i, fi in enumerate(feat_idx.tolist())}

# group_id → list of row_idx into A/B
group_to_rows = {}
for gid, grp in mem.groupby("ProductName"):
    rows = []
    for fi in grp["feat_idx"].dropna().astype(int):
        if fi in idx2row: rows.append(idx2row[fi])
    group_to_rows[gid] = sorted(set(rows))
print(f"[groups] with membership: {sum(len(v)>0 for v in group_to_rows.values())}/{len(group_to_rows)}")

# group-level amine vector (mean over candidate amines in that product group)
def amine_vec_for_group(gid: str):
    rows = group_to_rows.get(gid, [])
    if not rows: return None
    return A[rows].mean(axis=0)

# ================== examples ==================
def build_examples(labels: pd.DataFrame) -> pd.DataFrame:
    rows, skipped = [], 0
    for _, r in labels.iterrows():
        eid = r["enzyme_id"]; gid = r["group_id"]; y = float(r["y"])
        if eid not in enzyme2idx: skipped += 1; continue
        row_ids = group_to_rows.get(gid, [])
        if not row_ids: skipped += 1; continue
        Avec = amine_vec_for_group(gid)
        if Avec is None: skipped += 1; continue
        rows.append({
            "enzyme_id": eid,
            "group_id": gid,
            "y": math.log1p(y) if CFG["label_log1p"] else y,
            "enz_idx": enzyme2idx[eid],
            "amine_vec": Avec,     # [1024]
            "row_ids": row_ids,    # list of indices into A/B
        })
    df = pd.DataFrame(rows)
    print(f"[examples] {len(df)} built | skipped {skipped}")
    return df

examples = build_examples(labels_long)

# ================== dataset / loader ==================
class PoolDataset(Dataset):
    def __init__(self, df, E_mat, A_mat, B_mat):
        self.df = df.reset_index(drop=True)
        self.E  = E_mat; self.A = A_mat; self.B = B_mat
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        Evec = self.E[int(r["enz_idx"])].astype(np.float32)
        Avec = np.asarray(r["amine_vec"], dtype=np.float32)
        idxs = list(map(int, r["row_ids"]))
        A_k  = self.A[idxs].astype(np.float32)   # [k, 1024] (not used by current scorer)
        B_k  = self.B[idxs].astype(np.float32)   # [k, 512]
        y    = float(r["y"])
        return (torch.from_numpy(Evec), torch.from_numpy(Avec),
                torch.from_numpy(A_k),  torch.from_numpy(B_k),
                torch.tensor(y, dtype=torch.float32))

def collate(batch):
    E,A,Ak,Bk,Y = zip(*batch)
    E = torch.stack(E,0); A = torch.stack(A,0); Y = torch.stack(Y,0)
    return E, A, list(Ak), list(Bk), Y

# === SPLIT HELPERS (clear, undeniable checks) ===
def _row_keys(df):
    # Identity of a row: (enzyme_id, group_id). You can add 'y' if you want stricter identity.
    return set(map(tuple, df[["enzyme_id","group_id"]].itertuples(index=False, name=None)))

def group_splits_by_enzyme(df, val_frac=0.15, test_frac=0.15, seed=1337):
    groups = df["enzyme_id"].values
    gss_outer = GroupShuffleSplit(n_splits=1, test_size=test_frac, random_state=seed)
    tr_idx, te_idx = next(gss_outer.split(df, groups=groups))
    base_train = df.iloc[tr_idx].reset_index(drop=True)
    test_pool  = df.iloc[te_idx].reset_index(drop=True)

    gss_inner = GroupShuffleSplit(
        n_splits=1, test_size=val_frac/(1.0-test_frac), random_state=seed
    )
    tr2_idx, va_idx = next(gss_inner.split(base_train, groups=base_train["enzyme_id"].values))
    train_pool = base_train.iloc[tr2_idx].reset_index(drop=True)
    val_pool   = base_train.iloc[va_idx].reset_index(drop=True)

    # === SPLIT CHECKS: enzyme-level disjointness ===
    assert set(train_pool["enzyme_id"]).isdisjoint(val_pool["enzyme_id"])
    assert set(train_pool["enzyme_id"]).isdisjoint(test_pool["enzyme_id"])
    assert set(val_pool["enzyme_id"]).isdisjoint(test_pool["enzyme_id"])

    # === SPLIT CHECKS: row-level disjointness ===
    assert _row_keys(train_pool).isdisjoint(_row_keys(val_pool)),  "Row overlap: train vs val"
    assert _row_keys(train_pool).isdisjoint(_row_keys(test_pool)), "Row overlap: train vs test"
    assert _row_keys(val_pool).isdisjoint(_row_keys(test_pool)),   "Row overlap: val vs test"

    # === SPLIT CHECKS: sizes add up ===
    assert len(train_pool) + len(val_pool) + len(test_pool) == len(df), "Split sizes don't sum to total!"

    print(f"[split] enzymes train={train_pool['enzyme_id'].nunique()} "
          f"val={val_pool['enzyme_id'].nunique()} test={test_pool['enzyme_id'].nunique()}")
    print(f"[split] rows    train={len(train_pool)} val={len(val_pool)} test={len(test_pool)}")
    print("[catalog] A/B/group_to_rows are label-free feature catalogs shared across splits (no target leakage).")  # clarity
    print("[SPLIT OK] Disjoint by enzyme & row, sizes add up.")
    return train_pool, val_pool, test_pool

train_pool, val_pool, test_pool = group_splits_by_enzyme(
    examples, val_frac=CFG["val_frac"], test_frac=CFG["test_frac"], seed=CFG["seed"]
)

train_ds = PoolDataset(train_pool, E_mat, A, B)
val_ds   = PoolDataset(val_pool,   E_mat, A, B)
test_ds  = PoolDataset(test_pool,  E_mat, A, B)

train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True,  collate_fn=collate)
val_loader   = DataLoader(val_ds,   batch_size=CFG["batch_size"], shuffle=False, collate_fn=collate)
test_loader  = DataLoader(test_ds,  batch_size=CFG["batch_size"], shuffle=False, collate_fn=collate)

print(f"[split] rows train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}")

# ================== model ==================
class PairScorer(nn.Module):
    """Score each BA candidate: f_theta(E, A, B_k) -> y_{e,(a,Bk)} >= 0"""
    def __init__(self, d_prot, d_am, d_ba, hidden=512, depth=2, dropout=0.1):
        super().__init__()
        dims = [d_prot + d_am + d_ba] + [hidden]*(depth-1) + [1]
        layers=[]
        for i in range(len(dims)-2):
            layers += [nn.Linear(dims[i], dims[i+1]), nn.ReLU(), nn.Dropout(dropout)]
        layers += [nn.Linear(dims[-2], dims[-1])]
        self.mlp = nn.Sequential(*layers)
        self.softplus = nn.Softplus()

    def forward_one(self, E, A, Bk):
        if Bk.dim() == 1: Bk = Bk.unsqueeze(0)
        k = Bk.size(0)
        Eexp = E.unsqueeze(0).expand(k, -1)
        Aexp = A.unsqueeze(0).expand(k, -1)
        x = torch.cat([Eexp, Aexp, Bk], dim=-1)
        return self.softplus(self.mlp(x)).squeeze(-1)  # [k]

    def forward(self, E, A, Bk):
        return self.forward_one(E, A, Bk)

class Mixer(nn.Module):
    """Enzyme-conditioned attention over BA candidates."""
    def __init__(self, d_prot, d_am, d_ba, d_att=128):
        super().__init__()
        self.q = nn.Sequential(nn.Linear(d_prot + d_am, d_att), nn.Tanh())
        self.k = nn.Linear(d_ba, d_att, bias=False)

    def forward_one(self, E, A, Bk, lambda_entropy=1e-3):
        if Bk.dim() == 1: Bk = Bk.unsqueeze(0)
        q = self.q(torch.cat([E, A], dim=-1))  # [d_att]
        K = self.k(Bk)                         # [k, d_att]
        w = torch.softmax(K @ q, dim=0)        # [k]
        ent = -(w * (w.clamp_min(1e-8).log())).sum() * lambda_entropy
        return w, ent

    def forward(self, E, A, Bk, lambda_entropy=1e-3):
        return self.forward_one(E, A, Bk, lambda_entropy=lambda_entropy)

class PooledModel(nn.Module):
    def __init__(self, d_prot, d_am, d_ba):
        super().__init__()
        self.scorer = PairScorer(d_prot, d_am, d_ba, hidden=512, depth=2, dropout=0.1)
        self.mixer  = Mixer(d_prot, d_am, d_ba, d_att=128)

    def forward(self, E, A, Ak_list, Bk_list, lambda_entropy=1e-3):
        B = E.size(0)
        Yhat = []
        total_ent = E.new_tensor(0.0)
        for b in range(B):
            yk = self.scorer(E[b], A[b], Bk_list[b])                     # [k]
            w, ent = self.mixer(E[b], A[b], Bk_list[b], lambda_entropy)  # [k], scalar
            Yhat.append((w * yk).sum())
            total_ent = total_ent + ent
        return torch.stack(Yhat, 0), total_ent / max(1, B)

# ================== training ==================
def run_epoch(loader, train_mode: bool, model=None, opt=None):
    model.train(train_mode)
    y_true, y_pred = [], []
    tot_loss = tot_ent = 0.0
    for E,A,Ak_list,Bk_list,Y in loader:
        E = E.to(CFG["device"]).float()
        A = A.to(CFG["device"]).float()
        Ak_list = [x.to(CFG["device"]).float() for x in Ak_list]
        Bk_list = [x.to(CFG["device"]).float() for x in Bk_list]
        Y = Y.to(CFG["device"]).float()

        Yhat, ent = model(E, A, Ak_list, Bk_list, lambda_entropy=CFG["lambda_entropy"])
        loss_data = F.mse_loss(Yhat, Y)
        loss = loss_data + ent

        if train_mode:
            opt.zero_grad(); loss.backward()
            if CFG["grad_clip"]:
                nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
            opt.step()

        y_true.extend(Y.detach().cpu().numpy()); y_pred.extend(Yhat.detach().cpu().numpy())
        tot_loss += float(loss_data.item()) * len(Y); tot_ent += float(ent.item()) * len(Y)

    n = len(y_true)
    y_true = np.array(y_true); y_pred = np.array(y_pred)
    return dict(loss=tot_loss/max(1,n), ent=tot_ent/max(1,n),
                rmse=rmse(y_true, y_pred), mae=mae(y_true, y_pred), r=pearsonr(y_true, y_pred))

set_seeds(CFG["seed"])
d_prot = int(E_mat.shape[1]); d_am = A.shape[1]; d_ba = B.shape[1]
model = PooledModel(d_prot, d_am, d_ba).to(CFG["device"])
opt   = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG["epochs"])

class EarlyStopper:
    def __init__(self, patience=8, min_delta=1e-4):
        self.patience=patience; self.min_delta=min_delta; self.best=float("inf"); self.bad=0
    def step(self, metric: float) -> bool:
        if metric < (self.best - self.min_delta):
            self.best = metric; self.bad = 0; return True
        self.bad += 1; return False
    def should_stop(self) -> bool: return self.bad >= self.patience

early = EarlyStopper(patience=CFG["patience"], min_delta=CFG["min_delta"])
best = {"rmse": float("inf")}; best_ep = -1

# === NEW: checkpoint hygiene so we don't reuse an old model by accident ===
if os.path.exists(CFG["ckpt_path"]):
    os.remove(CFG["ckpt_path"])
    print(f"[ckpt] removed old checkpoint: {CFG['ckpt_path']}")

for ep in range(1, CFG["epochs"]+1):
    tr = run_epoch(train_loader, True,  model, opt)
    va = run_epoch(val_loader,   False, model, None)
    sched.step()
    print(f"[{ep:03d}] train RMSE {tr['rmse']:.3f} MAE {tr['mae']:.3f} r {tr['r']:.3f} | "
          f"val RMSE {va['rmse']:.3f} MAE {va['mae']:.3f} r {va['r']:.3f}  H(ent) {tr['ent']:.4f}")
    if early.step(va["rmse"]):
        best = va.copy(); best_ep = ep
        os.makedirs(os.path.dirname(CFG["ckpt_path"]) or ".", exist_ok=True)
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(),
                    "epoch": ep, "val": va, "cfg": CFG}, CFG["ckpt_path"])
        print(f"[ckpt] saved epoch {ep} with val RMSE {va['rmse']:.4f}")  # === NEW ===
    if early.should_stop():
        print(f"[early-stop] no improvement for {CFG['patience']} epochs. "
              f"Best val RMSE={best['rmse']:.3f} @ epoch {best_ep}.")
        break

# best-on-test
d = torch.load(CFG["ckpt_path"], map_location=CFG["device"])
print(f"[ckpt] loaded epoch {d.get('epoch')} with val RMSE {d.get('val',{}).get('rmse')}")  # === NEW ===
model.load_state_dict(d["model"])
te = run_epoch(test_loader, False, model, None)
print(f"[TEST] RMSE {te['rmse']:.3f} MAE {te['mae']:.3f} r {te['r']:.3f} "
      f"(best val RMSE {best['rmse']:.3f} @ ep {best_ep})")

# ================== Top-k attention inspector ==================
feat_idx_arr = feat_idx
sub_by_feat  = sub.set_index("feat_idx", drop=True)

def rowidx_to_info(row_idx: int) -> dict:
    fi = int(feat_idx_arr[row_idx])
    info = sub_by_feat.loc[fi]
    return dict(
        feat_idx=fi,
        parent_ba_name=str(info["parent_ba_name"]),
        amine_name=str(info["amine_name"]),
        ba_smiles=str(info.get("ba_smiles", "")),
        amine_smiles=str(info.get("amine_smiles","")),
    )

# === NEW: helper to treat y==0 as "no ranking shown" in inspection ===
def _is_zero_activity(y, eps=1e-8):
    # With log1p targets, zero intensity -> y == 0 exactly.
    return y <= eps

@torch.no_grad()
def topk_attention_for(model, enzyme_id: str, group_id: str, topk: int = 5):
    if enzyme_id not in enzyme2idx:
        raise KeyError(f"enzyme_id '{enzyme_id}' not in embeddings.")
    if group_id not in group_to_rows or len(group_to_rows[group_id]) == 0:
        raise KeyError(f"group_id '{group_id}' has no instances.")
    Evec = torch.from_numpy(E_mat[enzyme2idx[enzyme_id]]).float().to(CFG["device"])
    Avec_np = amine_vec_for_group(group_id)
    if Avec_np is None:
        raise KeyError(f"group_id '{group_id}' has no amine vector.")
    Avec = torch.from_numpy(Avec_np).float().to(CFG["device"])
    row_ids = group_to_rows[group_id]
    Bk = torch.from_numpy(B[row_ids]).float().to(CFG["device"])
    model.eval()
    yk = model.scorer(Evec, Avec, Bk)                      # [k]
    w, _ = model.mixer(Evec, Avec, Bk, lambda_entropy=0.0) # [k]
    contrib = (w * yk).detach().cpu().numpy()
    order = contrib.argsort()[::-1]
    recs = []
    for rank, j in enumerate(order[:topk], 1):
        info = rowidx_to_info(row_ids[j])
        recs.append({
            "rank": rank,
            "feat_idx": info["feat_idx"],
            "parent_ba_name": info["parent_ba_name"],
            "amine_name": info["amine_name"],
            "weight": float(w[j].detach().cpu()),
            "score": float(yk[j].detach().cpu()),
            "weight_x_score": float(contrib[j]),
        })
    return pd.DataFrame(recs)

def sanity_print_topk(model, df_subset: pd.DataFrame, topk: int = 5, n_examples: int = 5, seed: int = 1337):
    rng = np.random.default_rng(seed)
    if len(df_subset) == 0:
        print("[topk] empty subset"); return
    pick_idx = rng.choice(len(df_subset), size=min(n_examples, len(df_subset)), replace=False)
    for i in pick_idx:
        r = df_subset.iloc[int(i)]
        eid, gid, y = r["enzyme_id"], r["group_id"], float(r["y"])
        print(f"\n=== {eid} | {gid} | y={y:.6f} ===")
        if _is_zero_activity(y):                                  # === NEW ===
            print("(no ranking shown: label indicates zero activity)")  # === NEW ===
            continue                                              # === NEW ===
        try:
            df_top = topk_attention_for(model, eid, gid, topk=CFG["topk_show"])
            print(df_top.to_string(index=False))
        except Exception as ex:
            print(f"(no top-k available) {ex}")

# quick peek
sanity_print_topk(model, val_pool,  topk=CFG["topk_show"], n_examples=5, seed=CFG["seed"])
sanity_print_topk(model, test_pool, topk=CFG["topk_show"], n_examples=5, seed=CFG["seed"])


In [ ]:
print(f"[labels] total={len(labels_long)}")
print(f"[examples] usable={len(examples)} (skipped={len(labels_long)-len(examples)})")
print(f"[split] train={len(train_pool)} val={len(val_pool)} test={len(test_pool)}")

## Validating Between the Predicted Intensities

Clean raw and clean tables.

In [ ]:
# ================== CACHE PREDICTIONS & INSPECT (run AFTER training) ==================
# --- build averaged labels from labels_long exactly as trained on ---
labels_avg = labels_long.rename(columns={"y": "Intensity_avg"}).copy()
if CFG["label_log1p"]:
    labels_avg["y_avg_log1p"] = np.log1p(labels_avg["Intensity_avg"])
else:
    labels_avg["y_avg_log1p"] = labels_avg["Intensity_avg"]

# --- small helper: recompute a single pooled prediction (cheap forward pass) ---
@torch.no_grad()
def _predict_one(model, enzyme_id: str, group_id: str):
    if enzyme_id not in enzyme2idx:
        return np.nan, np.nan, 0
    idxs = group_to_rows.get(group_id, [])
    Avec_np = amine_vec_for_group(group_id)
    if not idxs or Avec_np is None:
        return np.nan, np.nan, 0
    Evec = torch.from_numpy(E_mat[enzyme2idx[enzyme_id]]).float().to(CFG["device"])
    Avec = torch.from_numpy(Avec_np).float().to(CFG["device"])
    Bk   = torch.from_numpy(B[idxs]).float().to(CFG["device"])
    model.eval()
    yk = model.scorer(Evec, Avec, Bk)                         # (k,)
    w, _ = model.mixer(Evec, Avec, Bk, lambda_entropy=0.0)    # (k,)
    yhat_log1p = float((w * yk).sum().detach().cpu())
    yhat_raw   = float(np.expm1(yhat_log1p)) if CFG["label_log1p"] else yhat_log1p
    return yhat_log1p, yhat_raw, len(idxs)

# --- cache predictions once per split; reuse later (no retraining needed) ---
@torch.no_grad()
def predict_split_df(model, df_split: pd.DataFrame, split_name: str) -> pd.DataFrame:
    rows = []
    for _, r in df_split.iterrows():
        eid, gid, y = r["enzyme_id"], r["group_id"], float(r["y"])
        ylp, yr, k = _predict_one(model, eid, gid)
        rows.append(dict(split=split_name, enzyme_id=eid, group_id=gid,
                         y=r["y"], yhat_log1p=ylp, yhat_raw=yr, k_candidates=k))
    return pd.DataFrame(rows)

print("[cache] computing predictions per split...")
pred_train_df = predict_split_df(model, train_pool, "train")
pred_val_df   = predict_split_df(model, val_pool,   "val")
pred_test_df  = predict_split_df(model, test_pool,  "test")
pred_all      = pd.concat([pred_train_df, pred_val_df, pred_test_df], ignore_index=True)
print("[cache] done. rows:", len(pred_all))

# --- (optional) raw rows table for trace-back to original LC-MS entries ---
raw_cols = {}
if "Enzyme" in heat_long.columns:      raw_cols["Enzyme"]      = "enzyme_id_raw"
if "ProductName" in heat_long.columns: raw_cols["ProductName"] = "group_id"
if "Intensity" in heat_long.columns:   raw_cols["Intensity"]   = "Intensity"
raw_base = heat_long.rename(columns=raw_cols).copy()
# normalize enzyme id just like training (strip "_repX")
def _norm_enzyme_id(s: str) -> str: return str(s).rsplit("_rep", 1)[0]
if "enzyme_id_raw" in raw_base.columns:
    raw_base["Intensity"] = pd.to_numeric(raw_base["Intensity"], errors="coerce")
    raw_base = raw_base.dropna(subset=["Intensity"]).copy()
    raw_base["enzyme_id"] = raw_base["enzyme_id_raw"].map(_norm_enzyme_id)

# --- top-k (optional) with zero-label suppression for display ---
def _is_zero_activity(y, eps=1e-8): return y <= eps

@torch.no_grad()
def topk_attention_for(model, enzyme_id: str, group_id: str, topk: int = 5) -> pd.DataFrame:
    if enzyme_id not in enzyme2idx:
        return pd.DataFrame()
    rows = group_to_rows.get(group_id, [])
    if not rows: return pd.DataFrame()
    Avec_np = amine_vec_for_group(group_id)
    if Avec_np is None: return pd.DataFrame()
    Evec = torch.from_numpy(E_mat[enzyme2idx[enzyme_id]]).float().to(CFG["device"])
    Avec = torch.from_numpy(Avec_np).float().to(CFG["device"])
    Bk   = torch.from_numpy(B[rows]).float().to(CFG["device"])
    model.eval()
    yk = model.scorer(Evec, Avec, Bk)                      # (k,)
    w, _ = model.mixer(Evec, Avec, Bk, lambda_entropy=0.0) # (k,)
    contrib = (w * yk).detach().cpu().numpy()
    order = contrib.argsort()[::-1]
    recs = []
    for rank, j in enumerate(order[:min(topk, len(order))], 1):
        fi = int(feat_idx[rows[j]])
        info = sub[sub["feat_idx"] == fi].iloc[0] if (sub["feat_idx"] == fi).any() else None
        recs.append({
            "rank": rank,
            "feat_idx": fi,
            "parent_ba_name": None if info is None else str(info.get("parent_ba_name","")),
            "amine_name":     None if info is None else str(info.get("amine_name","")),
            "weight": float(w[j].detach().cpu()),
            "score": float(yk[j].detach().cpu()),
            "weight_x_score": float(contrib[j]),
        })
    return pd.DataFrame(recs)

# --- one-stop inspector: averaged label + cached prediction (+ optional raw & top-k) ---
def inspect_enzyme(enzyme_id: str, which_split: str = None, show_raw=True, topk=5, max_raw_rows_per_group=10):
    """
    enzyme_id: normalized ID (after '_rep' stripping)
    which_split: None | 'train' | 'val' | 'test'
    """
    # choose pairs
    if which_split is None:
        pairs = labels_avg[labels_avg["enzyme_id"] == enzyme_id][["enzyme_id","group_id"]].copy()
        preds_here = pred_all[pred_all["enzyme_id"] == enzyme_id].copy()
    else:
        if which_split not in {"train","val","test"}:
            raise ValueError("which_split must be one of None, 'train','val','test'")
        base = {"train": train_pool, "val": val_pool, "test": test_pool}[which_split]
        pairs = base[base["enzyme_id"] == enzyme_id][["enzyme_id","group_id"]].copy()
        preds_here = {"train": pred_train_df, "val": pred_val_df, "test": pred_test_df}[which_split]
        preds_here = preds_here[preds_here["enzyme_id"] == enzyme_id].copy()

    if pairs.empty:
        print(f"[inspect] no (enzyme, group) pairs for enzyme_id='{enzyme_id}' in split={which_split}")
        return None

    view = pairs.merge(labels_avg, on=["enzyme_id","group_id"], how="left")
    view = view.merge(preds_here[["enzyme_id","group_id","yhat_log1p","yhat_raw","k_candidates"]],
                      on=["enzyme_id","group_id"], how="left")

    # mark if in examples (usable by the model)
    eg_flag = examples[["enzyme_id","group_id"]].drop_duplicates().copy()
    eg_flag["in_examples"] = True
    view = view.merge(eg_flag, on=["enzyme_id","group_id"], how="left").fillna({"in_examples": False})

    # pretty columns
    cols = ["enzyme_id","group_id","Intensity_avg","y_avg_log1p","yhat_raw","yhat_log1p","k_candidates","in_examples"]
    view = view[cols].sort_values(["group_id"]).reset_index(drop=True)

    print(f"\n[inspect] enzyme_id='{enzyme_id}'  split={which_split}")
    print(view.to_string(index=False))

    # optional raw snippet
    if show_raw and "enzyme_id" in raw_base.columns:
        rb = raw_base[raw_base["enzyme_id"] == enzyme_id].copy()
        if not rb.empty:
            # show a few raw rows per group
            pieces = []
            for gid, grp in rb.groupby("group_id"):
                g = grp.sort_index().head(max_raw_rows_per_group)
                g = g.assign(_note="raw_row").loc[:, ["_note","enzyme_id_raw","group_id","Intensity"]]
                pieces.append(g)
            raw_snips = pd.concat(pieces, axis=0) if pieces else pd.DataFrame(columns=["_note","enzyme_id_raw","group_id","Intensity"])
            print("\nRaw rows (first few per group):")
            print(raw_snips.to_string(index=False))

    # optional top-k (skip when label is zero for that group)
    if topk and topk > 0:
        print("\nTop-k contributions (skipped for groups with y_avg_log1p == 0):")
        for _, r in view.iterrows():
            if CFG["label_log1p"] and _is_zero_activity(float(r["y_avg_log1p"])) or (not CFG["label_log1p"] and r["Intensity_avg"] == 0):
                print(f"  - {r['group_id']}: (no ranking shown; zero label)")
                continue
            df_top = topk_attention_for(model, r["enzyme_id"], r["group_id"], topk=topk)
            if df_top.empty:
                print(f"  - {r['group_id']}: (no candidates / unavailable)")
            else:
                print(f"\n  - {r['group_id']}:")
                print(df_top.to_string(index=False))

    return view


In [ ]:
inspect_enzyme("A0A848U0C2")

# Or, restrict to only the groups for this enzyme in the validation split:
inspect_enzyme("A0A848U0C2", which_split="val")